In [1]:
### IMPORT EXTERNAL FUNCTIONS
import mne
import numpy as np
from os.path import join
import os
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import scipy

from functions import utils

# 1. Load synchronized session full stream #

In [ ]:
# Session to preprocess
session_id = "C009 mSST"

working_path = os.path.dirname(os.getcwd())
onedrive_path = utils._get_onedrive_path()
sub = session_id.split(' ') [0]

if session_id.startswith('sub'):
    condition = session_id.split(' ') [1] + ' ' + session_id.split(' ') [2]
    sub_onedrive_path_task = join(onedrive_path, sub, 'synced_data', session_id)
elif session_id.startswith('C'):
    condition = session_id
    sub_onedrive_path_task = join(onedrive_path, sub, 'synced_data', condition)

#  Set saving path
results_path = join(working_path, "results")
saving_path = join(results_path, "eeg_output", "single_sub", session_id)
if not os.path.isdir(saving_path):
    os.makedirs(saving_path)
fig_save_path = join(saving_path, "figures")
os.makedirs(fig_save_path, exist_ok=True)  # Create the directory if it doesn't exist

data_save_path = os.path.join(saving_path,"data")
os.makedirs(data_save_path, exist_ok=True)  # Create the directory if it doesn't exist

# Load raw data
filename = [f for f in os.listdir(sub_onedrive_path_task) if (
    f.endswith('.set') and f.startswith('SYNCHRONIZED_EXTERNAL'))]
file = join(sub_onedrive_path_task, filename[0])
raw = mne.io.read_raw(file, preload=True)


# 2. Annotate automatically and verify break identification #

In [ ]:
# t_start_after_previous defines how much time after the previous annotation the break should start
# t_stop_before_next defines how much time before the next annotation the break should stop
# = how much time without events do we want at beginning and end of each (in our case) blocks.

break_annot = mne.preprocessing.annotate_break(raw, min_break_duration=10, t_start_after_previous=4, t_stop_before_next=4)
raw_breaks = raw.set_annotations(raw.annotations + break_annot)

%matplotlib inline
raw_breaks.plot()

In [ ]:
# enter here manually number of blocks (3 or 4 for mSST usually):
n_blocks = 3

In [ ]:
# Get break start/end times
break_on = break_annot.onset
break_off = break_annot.onset + break_annot.duration

# Sort breaks in chronological order
# so that when we use them later to index breaks in the data, they are in the correct order
idx = np.argsort(break_on)
break_on = break_on[idx]
break_off = break_off[idx]

# Double check that breaks are in the right place (first break should be at start, second in middle etc.)
print(f"Break start times (s): {break_on}")
print(f"Break end times (s): {break_off}")

if n_blocks >= 3:
    # Define block windows
    t1_start, t1_stop = break_off[0], break_on[1]   # after break1 (=beginning of stream) until break2 (=halfway break) starts
    t2_start, t2_stop = break_off[1], break_on[2]   # after break2 (=halfway break) until break3 (=end of stream) starts
    t3_start, t3_stop = break_off[2], break_on[3]   # after break3 (=end of stream) until the end of recording

    print(f"Block 1: {t1_start:.1f} – {t1_stop:.1f} s")
    print(f"Block 2: {t2_start:.1f} – {t2_stop:.1f} s")
    print(f"Block 3: {t3_start:.1f} – {t3_stop:.1f} s")

if n_blocks == 4:
    t4_start, t4_stop = break_off[3], raw.times[-1] # after break4 (if there is one) until the end of recording
    print(f"Block 4: {t4_start:.1f} – {t4_stop:.1f} s")

# 3. Crop each block and save #

In [ ]:
# Save blocks
save_folder = join(saving_path, "raw_blocks") 
os.makedirs(save_folder, exist_ok=True)

if n_blocks >= 3:
    # Crop into three Raw objects
    raw_block1 = raw_breaks.copy().crop(tmin=t1_start, tmax=t1_stop, include_tmax=False)
    raw_block1.save(f"{save_folder}/{sub}_{condition}_mSST_block1_raw.fif", overwrite=True)

    raw_block2 = raw_breaks.copy().crop(tmin=t2_start, tmax=t2_stop, include_tmax=False)
    raw_block2.save(f"{save_folder}/{sub}_{condition}_mSST_block2_raw.fif", overwrite=True)

    raw_block3 = raw_breaks.copy().crop(tmin=t3_start, tmax=t3_stop, include_tmax=False)
    raw_block3.save(f"{save_folder}/{sub}_{condition}_mSST_block3_raw.fif", overwrite=True)

if n_blocks == 4:
    # Crop into a fourth Raw object
    raw_block4 = raw_breaks.copy().crop(tmin=t4_start, tmax=t4_stop, include_tmax=False)
    raw_block4.save(f"{save_folder}/{sub}_{condition}_mSST_block4_raw.fif", overwrite=True)

# Save block timings into .txt file
with open(f"{save_folder}/{sub}_{condition}_block_timings.txt", "w") as f:
    f.write(f"Block 1: {t1_start:.1f} – {t1_stop:.1f} s\n")
    f.write(f"Block 2: {t2_start:.1f} – {t2_stop:.1f} s\n")
    f.write(f"Block 3: {t3_start:.1f} – {t3_stop:.1f} s\n")
    if n_blocks == 4:
        f.write(f"Block 4: {t4_start:.1f} – {t4_stop:.1f} s\n")